# Elevator Dynamic Parameter Optimization using Genetic Algorithm

This notebook demonstrates the dynamic byte coding genetic algorithm for optimizing elevator dynamic parameters to minimize vertical vibration.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src import ElevatorDynamicModel, DynamicByteCodedGA

np.random.seed(42)

## 1. Define Elevator Parameters

Create a synthetic 2:1 traction-type elevator system with 9 degrees of freedom.

In [ ]:
# Initial elevator parameters
masses = np.array([800.0, 1200.0, 900.0, 1500.0, 1000.0, 1200.0])  # kg
inertias = np.array([100.0, 120.0, 110.0])  # kg·m²

# Initial stiffness and damping (these will be optimized)
stiffness_init = np.array([3e5, 2e5, 2.5e5, 2e5, 2.2e5, 2.3e5])  # N/m
damping_init = np.array([3e3, 2.5e3, 2.8e3, 2.5e3, 2.6e3, 2.7e3])  # N·s/m

# Create the model
model = ElevatorDynamicModel(masses, inertias, stiffness_init, damping_init)

print(f"Masses: {masses}")
print(f"Stiffness: {stiffness_init}")
print(f"Damping: {damping_init}")

## 2. Define Excitation Signal

Create a synthetic excitation representing elevator movement (acceleration/deceleration).

In [ ]:
# Generate excitation signal
t = np.linspace(0, 2, 500)
excitation = 50 * np.sin(2 * np.pi * 2 * t) * (1 + 0.3 * np.sin(2 * np.pi * 0.5 * t))

# Compute initial response
initial_response = model.compute_response(t, excitation)
initial_peak = model.objective_function(excitation)

print(f"Initial peak acceleration: {initial_peak:.4f} m/s²")

plt.figure(figsize=(12, 4))
plt.plot(t, excitation, linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Excitation Force (N)')
plt.title('Synthetic Excitation Signal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/excitation.png', dpi=100, bbox_inches='tight')
plt.close()

print(f"Excitation plot saved")

## 3. Run Genetic Algorithm Optimization

Optimize the stiffness and damping coefficients to minimize peak acceleration.

In [ ]:
# Define bounds for optimization
n_params = 6  # 6 stiffness/damping pairs
min_vals = np.array([1e5, 1e5, 1e5, 1e5, 1e5, 1e5])  # Minimum stiffness values
max_vals = np.array([8e5, 8e5, 8e5, 8e5, 8e5, 8e5])  # Maximum stiffness values

def objective_for_ga(params):
    """Objective function for GA: minimize peak acceleration."""
    model.stiffness = params[:6]
    return model.objective_function(excitation)

# Create and run GA
ga = DynamicByteCodedGA(population_size=30, generations=50, mutation_rate=0.15, crossover_rate=0.85)
optimal_params, fitness_history = ga.optimize(objective_for_ga, n_params, min_vals, max_vals)

print(f"Optimization completed!")
print(f"Initial peak acceleration: {initial_peak:.4f} m/s²")
print(f"Optimized peak acceleration: {fitness_history[-1]:.4f} m/s²")
print(f"Reduction: {(1 - fitness_history[-1] / initial_peak) * 100:.1f}%")

## 4. Verify Results with Optimized Model

In [ ]:
# Update model with optimized parameters
model_optimized = ElevatorDynamicModel(masses, inertias, optimal_params, damping_init)
optimized_response = model_optimized.compute_response(t, excitation)
optimized_peak = model_optimized.objective_function(excitation)

print(f"\nOptimized stiffness parameters:")
for i, k in enumerate(optimal_params):
    print(f"  k{i}: {k:.2e} N/m (initial: {stiffness_init[i]:.2e} N/m)")

print(f"\nComparison:")
print(f"  Peak acceleration (initial): {initial_peak:.4f} m/s²")
print(f"  Peak acceleration (optimized): {optimized_peak:.4f} m/s²")
print(f"  Improvement: {((initial_peak - optimized_peak) / initial_peak * 100):.1f}%")

## 5. Visualize Fitness Evolution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(fitness_history, linewidth=2, label='Best Fitness')
ax.axhline(y=optimized_peak, color='r', linestyle='--', linewidth=2, label=f'Final: {optimized_peak:.4f}')
ax.axhline(y=initial_peak, color='g', linestyle='--', linewidth=2, label=f'Initial: {initial_peak:.4f}')
ax.set_xlabel('Generation')
ax.set_ylabel('Peak Acceleration (m/s²)')
ax.set_title('Genetic Algorithm Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/fitness_evolution.png', dpi=100, bbox_inches='tight')
plt.close()

print("Fitness evolution plot saved")

## 6. Compare Vibration Responses

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t, initial_response, linewidth=2, label='Initial', alpha=0.7)
ax.plot(t, optimized_response, linewidth=2, label='Optimized', alpha=0.7)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Acceleration (m/s²)')
ax.set_title('Vertical Vibration Acceleration: Initial vs Optimized')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/vibration_comparison.png', dpi=100, bbox_inches='tight')
plt.close()

print("Vibration comparison plot saved")